In [1]:
import pandas as pd

In [2]:
df_depression = pd.read_csv("health/depression.csv")
df_drugs = pd.read_csv("health/drugs.csv")

In [3]:
df_depression.info()

<class 'pandas.DataFrame'>
RangeIndex: 6936 entries, 0 to 6935
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   population_group_id    6936 non-null   int64  
 1   population_group_name  6936 non-null   str    
 2   measure_id             6936 non-null   int64  
 3   measure_name           6936 non-null   str    
 4   location_id            6936 non-null   int64  
 5   location_name          6936 non-null   str    
 6   sex_id                 6936 non-null   int64  
 7   sex_name               6936 non-null   str    
 8   age_id                 6936 non-null   int64  
 9   age_name               6936 non-null   str    
 10  cause_id               6936 non-null   int64  
 11  cause_name             6936 non-null   str    
 12  metric_id              6936 non-null   int64  
 13  metric_name            6936 non-null   str    
 14  year                   6936 non-null   int64  
 15  val            

In [4]:
df_drugs.info()

<class 'pandas.DataFrame'>
RangeIndex: 6936 entries, 0 to 6935
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   population_group_id    6936 non-null   int64  
 1   population_group_name  6936 non-null   str    
 2   measure_id             6936 non-null   int64  
 3   measure_name           6936 non-null   str    
 4   location_id            6936 non-null   int64  
 5   location_name          6936 non-null   str    
 6   sex_id                 6936 non-null   int64  
 7   sex_name               6936 non-null   str    
 8   age_id                 6936 non-null   int64  
 9   age_name               6936 non-null   str    
 10  cause_id               6936 non-null   int64  
 11  cause_name             6936 non-null   str    
 12  metric_id              6936 non-null   int64  
 13  metric_name            6936 non-null   str    
 14  year                   6936 non-null   int64  
 15  val            

In [5]:
df_drugs.head()

,population_group_id,population_group_name,measure_id,measure_name,location_id,location_name,sex_id,sex_name,age_id,age_name,cause_id,cause_name,metric_id,metric_name,year,val,upper,lower
0,1,All Population,5,Prevalence,26,Papua New Guinea,3,Both,22,All ages,561,Drug use disorders,3,Rate,1990,701.603762,990.827097,490.764041
1,1,All Population,5,Prevalence,203,Cabo Verde,3,Both,22,All ages,561,Drug use disorders,3,Rate,1990,218.281838,274.220576,174.619417
2,1,All Population,5,Prevalence,28,Solomon Islands,3,Both,22,All ages,561,Drug use disorders,3,Rate,1990,627.135210,898.287022,438.134170
3,1,All Population,5,Prevalence,435,South Sudan,3,Both,22,All ages,561,Drug use disorders,3,Rate,1990,304.209179,440.781026,220.576091
4,1,All Population,5,Prevalence,85,Israel,3,Both,22,All ages,561,Drug use disorders,3,Rate,1990,703.489832,788.958621,627.316396


In [6]:
import pandas as pd
import numpy as np

# ============================================================
# 1) Generic cleaner for GBD/IHME-style health indicators
#    (works for depression, drugs, etc.)
# ============================================================
def clean_gbd_health_df(
    df: pd.DataFrame,
    indicator_name: str,
    min_year: int = 2000,
    missing_threshold: float = 30.0,
    # Filters (set to None to skip a filter)
    population_group_name: str | None = "All Population",
    sex_name: str | None = "Both",
    age_name: str | None = "All ages",
    metric_name: str | None = "Rate",
    measure_name: str | None = None,      # e.g., "Prevalence" if you want to enforce it
    cause_name: str | None = None          # e.g., "Drug use disorders" / "Depressive disorders"
):
    """
    Returns:
      - df_clean: columns = [country, year, <indicator_name>]
      - missing_stats: missing % by country (computed on observed rows, no artificial year expansion)
    """

    df = df.copy()

    # --- Ensure expected columns exist (fail early if not)
    needed = {
        "population_group_name", "sex_name", "age_name", "metric_name",
        "measure_name", "cause_name", "location_name", "year", "val"
    }
    missing_cols = needed - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing expected columns: {missing_cols}")

    # --- 1) Apply dimension filters to lock ONE consistent slice
    mask = pd.Series(True, index=df.index)

    if population_group_name is not None:
        mask &= (df["population_group_name"] == population_group_name)
    if sex_name is not None:
        mask &= (df["sex_name"] == sex_name)
    if age_name is not None:
        mask &= (df["age_name"] == age_name)
    if metric_name is not None:
        mask &= (df["metric_name"] == metric_name)
    if measure_name is not None:
        mask &= (df["measure_name"] == measure_name)
    if cause_name is not None:
        mask &= (df["cause_name"] == cause_name)

    df = df[mask].copy()

    # --- 2) Keep years >= min_year
    df = df[df["year"] >= min_year].copy()

    # --- 3) Keep only what we need (country, year, score)
    df = df[["location_name", "year", "val"]].rename(
        columns={"location_name": "country", "val": "score"}
    )

    # --- 4) If duplicates exist (same country-year), aggregate to one value
    # (GBD data usually shouldn't duplicate after filters, but safe)
    df = df.groupby(["country", "year"], as_index=False)["score"].mean()

    # --- 5) Missing % by country (based on observed rows; no artificial year grid)
    missing_stats = (
        df.groupby("country")["score"]
          .apply(lambda x: x.isna().mean() * 100)
          .reset_index(name="missing_percent")
    )

    # --- 6) Drop countries with > threshold missing
    countries_to_keep = missing_stats.loc[
        missing_stats["missing_percent"] <= missing_threshold, "country"
    ]
    df = df[df["country"].isin(countries_to_keep)].copy()

    # --- 7) Sort for time-series ops
    df = df.sort_values(["country", "year"]).reset_index(drop=True)

    # --- 8) Impute within each country (interpolate + fill ends)
    df["score"] = (
        df.groupby("country")["score"]
          .transform(lambda s: s.interpolate(method="linear", limit_direction="both"))
    )
    df["score"] = (
        df.groupby("country")["score"]
          .transform(lambda s: s.ffill().bfill())
    )

    # --- 9) Final column name
    df = df.rename(columns={"score": indicator_name})

    return df, missing_stats


# ============================================================
# 2) Cleaner specifically for DRUGS (your dataset shown)
# ============================================================
def clean_drugs_df(
    df_drugs: pd.DataFrame,
    indicator_name: str = "drug_use_rate",
    min_year: int = 2000,
    missing_threshold: float = 30.0
):
    return clean_gbd_health_df(
        df=df_drugs,
        indicator_name=indicator_name,
        min_year=min_year,
        missing_threshold=missing_threshold,
        population_group_name="All Population",
        sex_name="Both",
        age_name="All ages",
        metric_name="Rate",
        measure_name="Prevalence",            # adjust if your drugs data uses another measure
        cause_name="Drug use disorders"       # based on your sample
    )


# ============================================================
# 3) Cleaner specifically for DEPRESSION
#    (same logic; just change cause_name, and keep/adjust measure_name)
# ============================================================
def clean_depression_df(
    df_depression: pd.DataFrame,
    indicator_name: str = "depression_rate",
    min_year: int = 2000,
    missing_threshold: float = 30.0,
    depression_cause_name: str = "Depressive disorders"   # adjust if your dataset uses a different label
):
    return clean_gbd_health_df(
        df=df_depression,
        indicator_name=indicator_name,
        min_year=min_year,
        missing_threshold=missing_threshold,
        population_group_name="All Population",
        sex_name="Both",
        age_name="All ages",
        metric_name="Rate",
        measure_name="Prevalence",            # adjust if depression data uses Incidence / DALYs / etc.
        cause_name=depression_cause_name
    )


# ============================================================
# 4) Example usage
# ============================================================
# Drugs
df_drugs_clean, drugs_missing = clean_drugs_df(df_drugs, indicator_name="drug_use_rate")

# Depression
df_dep_clean, dep_missing = clean_depression_df(df_depression, indicator_name="depression_rate")

# Quick checks
print(df_drugs_clean.head())
print(df_drugs_clean["country"].nunique(), df_drugs_clean["year"].min(), df_drugs_clean["year"].max())


       country  year  drug_use_rate
0  Afghanistan  2000     401.444325
1  Afghanistan  2001     391.480996
2  Afghanistan  2002     385.336941
3  Afghanistan  2003     392.325879
4  Afghanistan  2004     401.691043
204 2000 2023


In [7]:
print(df_dep_clean.head())
print(df_dep_clean["country"].nunique(), df_dep_clean["year"].min(), df_dep_clean["year"].max())

       country  year  depression_rate
0  Afghanistan  2000      2798.567090
1  Afghanistan  2001      2788.721229
2  Afghanistan  2002      2761.116618
3  Afghanistan  2003      2758.209011
4  Afghanistan  2004      2751.423442
204 2000 2023


In [8]:
# Merge por country y year
df_mental_health = pd.merge(
    df_drugs_clean,
    df_dep_clean,
    on=["country", "year"],
    how="inner"
)

# Ver resultados
print("Shape:", df_mental_health.shape)
print("Países únicos:", df_mental_health["country"].nunique())
print("Años únicos:", df_mental_health["year"].nunique())

df_mental_health.head()

Shape: (4896, 4)
Países únicos: 204
Años únicos: 24


,country,year,drug_use_rate,depression_rate
0,Afghanistan,2000,401.444325,2798.567090
1,Afghanistan,2001,391.480996,2788.721229
2,Afghanistan,2002,385.336941,2761.116618
3,Afghanistan,2003,392.325879,2758.209011
4,Afghanistan,2004,401.691043,2751.423442


In [ ]:
df_mental_health.to_parquet("cleaned_data/df_mental.parquet", index=False)